# FigureVault on Google Colab
This notebook sets up the environment to run the FigureVault pipeline on Google Colab.
**Before running:** Make sure to go to `Runtime` -> `Change runtime type` and select **T4 GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

# Clean up any previous broken runs
shutil.rmtree('/content/figureVault', ignore_errors=True)
shutil.rmtree('/content/figureVault_temp', ignore_errors=True)

# Unzip directly to target folder
!unzip -q -o "/content/drive/MyDrive/figureVault.zip" -d /content/figureVault

# Fix nesting if the zip had a root 'figureVault' folder inside it
nested = '/content/figureVault/figureVault'
if os.path.exists(nested):
    for item in os.listdir(nested):
        shutil.move(os.path.join(nested, item), '/content/figureVault/')
    os.rmdir(nested)

%cd /content/figureVault
print("Directory contents:")
!ls -F

In [ ]:
# Install python dependencies
!pip install -r requirements.txt
!pip install watchdog  # Recommended for Streamlit on Linux

# Install system dependencies
!apt-get update
!apt-get install -y libgl1-mesa-glx pciutils lshw zstd default-jdk

### Build PDFFigures2 from source
Required for high-accuracy bounding box extraction.

In [ ]:
# Install sbt (Scala Build Tool)
!apt-get install apt-transport-https curl gnupg -yqq
!echo "deb https://repo.scala-sbt.org/scalasbt/debian all main" | sudo tee /etc/apt/sources.list.d/sbt.list
!echo "deb https://repo.scala-sbt.org/scalasbt/debian /" | sudo tee /etc/apt/sources.list.d/sbt_old.list
!curl -sL "https://keyserver.ubuntu.com/pks/lookup?op=get&search=0x2EE0EA64E40A89B84B2DF73499E82A75642AC823" | sudo -H gpg --no-default-keyring --keyring gnupg-ring:/etc/apt/trusted.gpg.d/scalasbt-release.gpg --import
!chmod 644 /etc/apt/trusted.gpg.d/scalasbt-release.gpg
!apt-get update
!apt-get install sbt -yqq

# Clone and build PDFFigures2
!git clone https://github.com/allenai/pdffigures2.git /content/pdffigures2
%cd /content/pdffigures2
!sbt assembly

# Copy the built JAR
!mkdir -p /content/figureVault/bin
!cp pdffigures2.jar /content/figureVault/bin/pdffigures2.jar
%cd /content/figureVault
print("✅ PDFFigures2 successfully built!")

In [ ]:
# Install & Start Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

process = subprocess.Popen(['ollama', 'serve'])
print("Ollama server starting...")
time.sleep(10)  # Wait for startup

!ollama pull gemma4:e4b

In [ ]:
# Run tests
!python -m pytest tests/

In [ ]:
# Run FigureVault help
!python main.py --help

### Run Streamlit UI

In [ ]:
import time
import urllib.request

# Kill any existing processes
!pkill -f streamlit
!pkill -f cloudflared

# Start Streamlit using nohup to prevent it from being killed when the cell moves on
!nohup streamlit run ui/app.py --server.port 8501 --server.address 127.0.0.1 > /content/logs.txt 2>&1 &

print("Waiting for Streamlit to start up...")
streamlit_up = False
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health')
        streamlit_up = True
        break
    except:
        time.sleep(1)

if not streamlit_up:
    print("❌ Streamlit failed to start. Printing logs:")
    !cat /content/logs.txt
else:
    print("✅ Streamlit is running!")
    
    # Download cloudflared
    !wget -q -O - https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 > /tmp/cloudflared
    !chmod +x /tmp/cloudflared
    
    print("\n\033[1;32m👉 Click the link ending in '.trycloudflare.com' below to open the UI!\033[0m\n")
    
    # Expose via Tunnel
    !/tmp/cloudflared tunnel --url http://127.0.0.1:8501